# 🧠 DietBot: Question Generation from Grounded PDF Content

This notebook is part of the **DietBot Project**, focused on generating clinically-grounded, answerable questions from authoritative sources such as the American Diabetes Association (ADA) guidelines.

## 📌 Purpose
To ingest PDF documents containing nutrition therapy guidelines for diabetes and:
- Parse and chunk the content for semantic retrieval.
- Embed the content using OpenAI embeddings.
- Generate a vector store and retriever for querying.
- Use a language model to generate diverse, realistic Q&A pairs.

## 📂 Inputs
- PDF documents located in `data/input_pdfs/`
- Example: `dci190009.pdf`, `dci190014.pdf`

## 📤 Outputs
- CSV files containing auto-generated questions stored in `data/csv_outputs/`
- These are used in subsequent steps like baseline generation and fact-checking.

---

🔧 Before running, ensure your `.env` file is configured (e.g., OpenAI key) and input PDFs are present in the correct folder.

## 🔐 Environment Configuration

Create a `.env` file in the root of your project to securely store your OpenAI API key.

### 📄 Example `.env` file

```env
OPENAI_API_KEY=sk-XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

## 📦 1. Environment Setup

This notebook loads required libraries for PDF-based question generation using LangChain.  
Make sure you've installed the following dependencies before continuing:

```bash
pip install langchain openai faiss-cpu pandas


In [1]:
# Core libraries
import os
import re
from datetime import datetime
import pandas as pd

# LangChain components
from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS


## 📄 Load and Embed PDFs

This block loads the ADA PDF files from `data/input_pdfs/`, splits them into overlapping chunks, and embeds them into a FAISS vectorstore using OpenAI embeddings. The retriever is used later for grounded question generation.

- ✅ Verifies that each expected PDF exists.
- 📚 Uses LangChain’s `RecursiveCharacterTextSplitter` to chunk documents.
- 🔍 Embeds chunks for semantic search via `OpenAIEmbeddings` and stores them in FAISS.


In [23]:
import os
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

# 📁 Define path to PDF input folder
pdf_dir = os.path.join("data", "input_pdfs")

# 📄 Collect all PDFs in the directory
pdf_files = [f for f in os.listdir(pdf_dir) if f.lower().endswith(".pdf")]
if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in directory: {pdf_dir}")

print(f"📚 Found {len(pdf_files)} PDF files:")
for f in pdf_files:
    print(f" - {f}")

# 🔄 Load PDFs
loaders = [PyPDFLoader(os.path.join(pdf_dir, file)) for file in pdf_files]
docs = sum([loader.load() for loader in loaders], [])
print(f"📄 Loaded total {len(docs)} documents (pages).")

# ✂️ Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"🔹 Split into {len(chunks)} chunks for embedding.")

# 🔗 Embed and index
vectorstore = FAISS.from_documents(chunks, OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

# 🔍 Optional: Retrieve sample context
reference_docs = retriever.get_relevant_documents("nutrition therapy diabetes")
pdf_context = "\n\n".join(doc.page_content for doc in reference_docs[:8])
print(f"✅ Sample retrieved context for query: {len(reference_docs[:8])} chunks.")


📚 Found 2 PDF files:
 - dci190009.pdf
 - dci190014.pdf
📄 Loaded total 28 documents (pages).
🔹 Split into 451 chunks for embedding.
✅ Sample retrieved context for query: 4 chunks.


## Structured Prompt for Question Generation

This prompt is designed to guide an LLM (e.g. GPT-4o) to generate **100 natural-sounding, emotionally grounded questions** related to diet and diabetes. It uses the embedded ADA guideline content as its sole source of truth for answerable questions.

### 🟣 Purpose
Generate realistic, emotionally honest questions from everyday people living with or learning about diabetes — focusing on diet, food choices, and the emotional side of managing blood sugar.

### 🟢 Audience
These questions should reflect the voices of:
- Newly diagnosed individuals (prediabetes or type 2 diabetes)
- People trying to manage diabetes via diet/lifestyle
- Caregivers of someone with diabetes
- Confused, overwhelmed individuals looking for practical guidance

These are **not** clinicians. They search Reddit, ask Google, and think out loud like:
> “Can I still eat pizza if I’m prediabetic?”

### 🟡 Role of the LLM
Act as a compassionate, health-literate AI.  
**Don’t** explain or answer anything — just **generate questions** that sound real, curious, and human.

### 🔵 Action
Generate 100 total questions, broken into:

#### ✅ 1–50: Answerable Questions
- Must be **strictly grounded** in the reference PDFs
- Focus on diet, glycemic control, fiber, weight loss, meal planning
- No medications, supplements, or non-ADA content
- Example:  
  - *"Do I have to stop eating bread if I’m prediabetic?"*  
  - *"What kind of diet helps bring A1C down naturally?"*

#### 🚫 51–100: Unanswerable / Speculative Questions
- Should reflect **common confusion, rumors, or misinformation**
- Not supported by the PDF content
- Example:  
  - *"Is intermittent fasting better than carbs control?"*  
  - *"Does cinnamon reverse insulin resistance?"*




In [10]:
# Step 4: Stronger Prompt for PDF Grounding and Natural Language Tone
diabetes_GenQ_prompt = f"""
📚 ADA Reference Documents (used for grounding Answerable Questions):
{pdf_context}

🟣 PURPOSE  
Generate 100 realistic, emotionally honest questions from everyday people who are diabetic or prediabetic. Focus heavily on diet, food choices, confusion about eating habits, and emotional concerns around managing blood sugar.

🟢 AUDIENCE  
These are regular adults who may be:
- Just diagnosed with prediabetes or type 2 diabetes
- Trying to manage diabetes through diet and lifestyle
- Caring for a parent or spouse with diabetes
- Struggling to understand how food affects their blood sugar

They are not doctors. They read Reddit, search symptoms on Google, and ask questions out loud like: “Can I still eat pizza if I’m prediabetic?” or “Is fruit bad now?”

🟡 ROLE  
You are a health-savvy, compassionate AI tasked with generating user questions to help train an empathetic diet-focused chatbot. Your goal is **not** to explain anything — just write questions that sound human.

🔵 ACTION  
Create 100 total questions, split into two labeled sections:

---

1. **Answerable Questions (50)**  
These must be grounded **strictly in the ADA reference material above**.  
Focus on diet, nutrition therapy, food confusion, weight loss, glycemic control, fiber, meal planning, etc.

Use only information clearly present in the PDFs.  
Do **not** include medications, supplements, tech apps, cultural trends, or your own general knowledge.

Example good questions:
- “Do I have to stop eating bread if I’m prediabetic?”
- “Is fruit okay or does it spike my sugar?”
- “What kind of diet helps bring A1C down naturally?”
- “Can losing weight actually reverse type 2 diabetes?”

---

2. **Unanswerable/Speculative Questions (50)**  
These should reflect common *confusion or misinformation* that **is not answered in the PDFs**.

Feel free to include:
- Rumors (e.g., “Cinnamon cures diabetes?”)
- Speculation (e.g., “Does keto work better than Metformin?”)
- Confusion (e.g., “Can my diabetes go away if I just eat clean?”)

Example unanswerable questions:
- “Is intermittent fasting better than carbs control?”
- “Can walking barefoot reduce blood sugar?”
- “Should I drink vinegar before meals?”
- “Does cinnamon reverse insulin resistance?”

---

🟠 OUTPUT FORMAT  
Answerable Questions (50):
1. ...
2. ...
...

Unanswerable Questions (50):
51. ...
52. ...
...

Keep each question short, casual, emotionally authentic — like someone typing into a chatbot or talking to a friend. Avoid medical tone or long complex sentences.
"""

## 🤖 Step 5–7: Generate, Parse, and Save Questions

- Calls GPT-4o with the structured prompt to generate 100 questions.
- Parses the response into labeled sections: **Answerable** and **Unanswerable**.
- Assigns each question a unique `Question ID` (e.g., Q001, Q002...).
- Saves the output as a timestamped CSV in the notebook directory.

📤 The CSV will be used in downstream evaluation and fact-checking notebooks.


In [12]:
# Step 5: Generate with GPT-4o
llm = ChatOpenAI(model="gpt-4o", temperature=0.7)
response = llm.invoke(diabetes_GenQ_prompt)

# Step 6: Parse response and assign IDs
raw_lines = response.content.strip().split("\n")
rows = []
section = None
idx = 1

for line in raw_lines:
    line = line.strip()
    if not line:
        continue

    if "Answerable Questions" in line:
        section = "Answerable"
        continue
    if "Unanswerable" in line or "Speculative" in line:
        section = "Unanswerable"
        continue

    match = re.match(r"^\d+[\).]\s*(.*)", line)
    if section and match:
        question = match.group(1).strip().strip('"')
        qid = f"Q{idx:03}"
        rows.append({
            "Question ID": qid,
            "Question Type": section,
            "Question": question
        })
        idx += 1

# Step 7: Save to CSV
df = pd.DataFrame(rows)

# Define output directory
output_dir = os.path.join("data", "csv_outputs")
os.makedirs(output_dir, exist_ok=True)  # Create if it doesn't exist

# Define filename with timestamp
filename = f"00_diabetes_qna_generated_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
filepath = os.path.join(output_dir, filename)

# Save to CSV
df.to_csv(filepath, index=False)
print(f"✅ Saved {len(df)} questions to {filepath}")


✅ Saved 100 questions to data/csv_outputs/00_diabetes_qna_generated_20250719_1145.csv


## ✅ Function: `is_answerable_by_pdfs_gpt()`

This function performs automated verification to ensure a question labeled "Answerable" is actually supported by the context found in the embedded PDF content.

### 🔍 What It Does
- **Skips** any questions labeled as `"Unanswerable"` (no need to check).
- Retrieves top `n_docs` relevant PDF chunks using a LangChain retriever.
- Builds a strict prompt asking a judge LLM (e.g., GPT-4o) to verify whether the question can be answered *clearly and fully* based **only** on the retrieved content.
- Returns a decision (`Yes`, `No`, or `Skipped`) and a short justification.

### 🧪 Why It's Important
This check ensures quality control. It filters out hallucinated or weakly-supported questions that could mislead users or violate the grounding intent of the DietBot system.

### 🛠️ Parameters
- `row`: A single row from a DataFrame with columns: `Question`, `Question Type`, and `Question ID`.
- `retriever`: LangChain-compatible retriever object over PDF chunks.
- `llm`: LLM (e.g., GPT-4o) used for verification, must support `.invoke(prompt)`.
- `n_docs`: Number of top documents to retrieve for grounding (default: 3).
- `verbose`: If `True`, prints the full prompt and response.

### 📤 Returns
A Pandas `Series` with:
1. `"Yes"`, `"No"`, `"Skipped"`, or `"Error"`
2. A short explanation or reason

---

**Note:** This function is a core part of the evaluation pipeline to ensure that generated questions truly reflect what is present in the authoritative source material.


In [25]:
def is_answerable_by_pdfs_gpt(row, retriever, llm, n_docs=3, verbose=False):
    question = row.get("Question", "")
    question_type = row.get("Question Type", "")
    question_id = row.get("Question ID", f"Q{row.name + 1:03d}")

    if question_type.lower() != "answerable":
        return pd.Series(["Skipped", "Question marked unanswerable, not checked.", ""])

    try:
        start = time.time()
        docs = retriever.get_relevant_documents(question)
        context = "\n\n".join(doc.page_content for doc in docs[:n_docs]).strip()

        if not context:
            print(f"❌ {question_id} | No context found.")
            return pd.Series(["No", "No relevant context retrieved.", ""])

        prompt = f"""
You are an expert fact-checking assistant. Your task is to determine whether the following question
can be accurately and clearly answered based *only* on the provided reference documents.

Be strict: if the information needed to answer the question is vague, missing, or implied without direct support, respond "No".

### Question:
{question}

### Reference Documents:
{context}

Based on the above content, is the question fully and clearly answerable? Respond with:

{{
  "answerable": "Yes" or "No",
  "reason": "<concise justification referencing the docs>",
  "citation": "<optional reference text or document snippet>"
}}
"""

        response = llm.invoke(prompt)
        parsed = json.loads(response.content.strip())
        elapsed = round(time.time() - start, 2)

        print(f"✅ {question_id} | Verified: {parsed['answerable']} | Time: {elapsed}s")

        if verbose:
            print("Prompt:\n", prompt)
            print("Response:\n", response.content)

        return pd.Series([
            parsed.get("answerable", "Error"),
            parsed.get("reason", ""),
            parsed.get("citation", "")
        ])

    except Exception as e:
        print(f"❌ {question_id} | Error: {str(e)}")
        return pd.Series(["Error", f"Exception: {str(e)}", ""])


## ✅ Step 8: Verify and Summarize Answerability

This step checks whether each **Answerable** question is truly grounded in the PDF content using an LLM-based fact-checker.

- Uses the `is_answerable_by_pdfs_gpt()` function to evaluate each question.
- Records the result (`Yes`, `No`, `Skipped`, `Error`) and a brief reason.
- Prints a summary of outcomes for quick inspection.
- Saves the summary as a separate CSV in `data/csv_outputs/`.

This helps assess the quality and factual grounding of the generated questions.


In [26]:
df[["LLMAnswerableCheck", "LLMCheckReason", "LLMCitedEvidence"]] = df.apply(
    lambda row: is_answerable_by_pdfs_gpt(row, retriever, llm),
    axis=1
)

# 🔍 Summarize results
summary = df["LLMAnswerableCheck"].value_counts().to_dict()
print("\n🧾 Verification Summary:")
for label in ["Yes", "No", "Skipped", "Error"]:
    count = summary.get(label, 0)
    print(f"- {label}: {count}")

# Optional: export summary as a note
summary_row = pd.DataFrame([summary])
summary_filename = f"00_qna_verification_summary_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
summary_filepath = os.path.join("data", "csv_outputs", summary_filename)
summary_row.to_csv(summary_filepath, index=False)
print(f"\n✅ Summary saved to {summary_filepath}")

✅ Q001 | Verified: Yes | Time: 4.44s
✅ Q002 | Verified: Yes | Time: 2.94s
✅ Q003 | Verified: Yes | Time: 2.74s
✅ Q004 | Verified: No | Time: 1.93s
✅ Q005 | Verified: Yes | Time: 3.04s
✅ Q006 | Verified: Yes | Time: 2.68s
✅ Q007 | Verified: No | Time: 2.64s
✅ Q008 | Verified: Yes | Time: 3.56s
✅ Q009 | Verified: No | Time: 1.9s
✅ Q010 | Verified: No | Time: 1.92s
✅ Q011 | Verified: Yes | Time: 2.08s
✅ Q012 | Verified: No | Time: 2.63s
✅ Q013 | Verified: No | Time: 1.27s
✅ Q014 | Verified: Yes | Time: 2.0s
✅ Q015 | Verified: No | Time: 1.83s
✅ Q016 | Verified: Yes | Time: 1.67s
✅ Q017 | Verified: No | Time: 2.17s
✅ Q018 | Verified: No | Time: 2.32s
✅ Q019 | Verified: Yes | Time: 2.01s
✅ Q020 | Verified: No | Time: 1.94s
✅ Q021 | Verified: No | Time: 2.5s
✅ Q022 | Verified: No | Time: 2.27s
✅ Q023 | Verified: No | Time: 4.69s
✅ Q024 | Verified: No | Time: 2.17s
✅ Q025 | Verified: No | Time: 1.63s
✅ Q026 | Verified: No | Time: 4.6s
✅ Q027 | Verified: No | Time: 2.73s
✅ Q028 | Verified: No 